# 🧩 Mini-Lab: Data-Shaped Failures, Including the Silent Ones

**Module: Debugging Lab 1 — Read the Failure** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Recognise** failures that produce no traceback at all — the code runs, prints a result, and the result is wrong
2. **Apply** the four-step debugging decision tree (reproduce → localise → shrink → change one thing) to silent failures
3. **Maintain** a debug log that records the error, root cause, and fix for every failure you encounter

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Silent Failure | An empty filter result or coerced dtype can silently change what your code is operating on, producing confident, wrong output without any exception or traceback. |
| Debugging Decision Procedure | A repeatable debugging procedure: reproduce the failure, localise it to one pipeline box, shrink the input, then change one thing at a time until the cause is clear. |


## Why Silent Failures Are Dangerous

In the previous mini-labs, every bug announced itself with a **traceback** — `TypeError`, `ModuleNotFoundError`, `ImportError`. You could read the error, localise the line, and fix it.

Silent failures are worse. The code finishes. Python is happy. **You** are happy — until you notice the output is empty, or the average is wrong, or the report has zero rows. There is no red text, no stack trace, nothing to Google.

The same four-step debug tree still works, but Step 1 ("reproduce") now means: **"Can I demonstrate the output is wrong?"** — and that requires knowing what the **correct** output should be.

We'll walk through three classic silent failures.

## Setup

In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

# Debug log — we'll append after each fix
debug_log = []

---

## Failure 1 — Empty Filter (String vs. Integer Comparison)

A common scenario: you load data from a CSV or JSON source and filter by a numeric field. Run the cell as written — the comparison is the bug.

### The Buggy Code

In [2]:
# Order data — imagine this came from a CSV (all values are strings)
orders = [
    {"id": "1", "product": "Widget",  "quantity": "10", "status": "shipped"},
    {"id": "2", "product": "Gadget",  "quantity": "5",  "status": "pending"},
    {"id": "3", "product": "Gizmo",   "quantity": "20", "status": "shipped"},
    {"id": "4", "product": "Doohick", "quantity": "3",  "status": "shipped"},
]

# Find large orders (quantity > 8)
# The comparison is the bug. Catch TypeError so Execute All can continue.
try:
    large_orders = [o for o in orders if o["quantity"] > 8]
    print(f"Large orders found: {len(large_orders)}")
    for o in large_orders:
        print(f"  {o['product']}: qty {o['quantity']}")
except TypeError:
    import traceback
    print(traceback.format_exc())

Traceback (most recent call last):
  File "C:\Users\molak\AppData\Local\Temp\ipykernel_42132\1127479031.py", line 12, in <module>
    large_orders = [o for o in orders if o["quantity"] > 8]
                                         ^^^^^^^^^^^^^^^^^
TypeError: '>' not supported between instances of 'str' and 'int'



That `TypeError` is real — Python 3 will not compare `"10"` and `8`. After you continue, we look at the **silent** cousin of the same bug: both sides as strings, no crash, still wrong.

### Step 1 — REPRODUCE

> *"Can I demonstrate the output is wrong?"*

A common next attempt after the TypeError is to quote the threshold so both sides are strings. Widget (qty 10) and Gizmo (qty 20) should still appear. If the result is **0**, that is the silent failure.

> 🧩 **Silent failure.** The no-traceback, wrong-output case from the video. Spot it here.

In [3]:
# Same filter, both sides strings — TypeError is gone, answer is still wrong
large_orders = [o for o in orders if o["quantity"] > "8"]

expected_count = 2   # Widget (10) and Gizmo (20) should match qty > 8
actual_count = len(large_orders)

print(f"Expected: {expected_count} large orders")
print(f"Actual:   {actual_count} large orders")
print()
if actual_count != expected_count:
    print("Reproduced. Output is silently wrong — no traceback, wrong result.")

Expected: 2 large orders
Actual:   0 large orders

Reproduced. Output is silently wrong — no traceback, wrong result.


### Step 2 — LOCALISE

> *"Which line is producing the wrong result?"*

The TypeError named the comparison. For the silent version there is no traceback, so inspect the values at that same point.

In [4]:
# Check what the comparison actually sees
for o in orders:
    qty = o["quantity"]
    print(f"  {o['product']}: quantity = {qty!r}  (type: {type(qty).__name__}),  qty > '8' → {qty > '8'}")

  Widget: quantity = '10'  (type: str),  qty > '8' → False
  Gadget: quantity = '5'  (type: str),  qty > '8' → False
  Gizmo: quantity = '20'  (type: str),  qty > '8' → False
  Doohick: quantity = '3'  (type: str),  qty > '8' → False


The `quantity` values are **strings**. `"10" > 8` is the loud `TypeError` you just saw. `"10" > "8"` is lexicographic: `"10" < "8"` because `"1"` comes before `"8"`. Widget and Gizmo both fail. Zero matches, no traceback.

**Root cause:** CSV-imported data arrives as strings; the code compared those strings as if they were integers.

### Step 3 — SHRINK

In [5]:
# Minimal case: one item that should pass but doesn't
single = [{"product": "Widget", "quantity": "10"}]
result = [o for o in single if int(o["quantity"]) > 8]   # with fix
bad    = [o for o in single if o["quantity"] > "8"]      # string compare — wrong!

print(f"With int():    {len(result)} match  (correct)")
print(f"String vs str: {len(bad)} match    ('10' < '8' lexicographically!)")

With int():    1 match  (correct)
String vs str: 0 match    ('10' < '8' lexicographically!)


### Step 4 — CHANGE ONE THING

Convert `quantity` to `int` before comparing.

In [6]:
# The ONE change: int() conversion
large_orders_fixed = [o for o in orders if int(o["quantity"]) > 8]

print(f"Large orders found: {len(large_orders_fixed)}")
for o in large_orders_fixed:
    print(f"  {o['product']}: qty {o['quantity']}")

Large orders found: 2
  Widget: qty 10
  Gizmo: qty 20


In [7]:
debug_log.append({
    "error": "Silent: filter returns 0 results, no traceback",
    "cause": "Quantity field is a string (from CSV); string comparison gives wrong results",
    "fix": "Convert to int() before comparing: int(o['quantity']) > 8",
})

---

## Failure 2 — Coerced Dtype (Silent Numeric Truncation)

Another classic: you compute a result and the answer **looks** reasonable, but it's wrong. Integer division silently truncates in some patterns.

### The Buggy Code

In [8]:
# Student scores — compute percentage
scores = [
    {"name": "Alice", "correct": 7, "total": 10},
    {"name": "Bob",   "correct": 3, "total": 10},
    {"name": "Carol", "correct": 9, "total": 10},
]

def compute_percentages(data):
    """Add a 'pct' field with the percentage score."""
    for row in data:
        row["pct"] = int(row["correct"] / row["total"] * 100)
    return data

results = compute_percentages(scores)
for r in results:
    print(f"  {r['name']}: {r['pct']}%")

  Alice: 70%
  Bob: 30%
  Carol: 90%


That looks right! 70%, 30%, 90%. But let's test with a trickier case.

In [9]:
# Edge case data
edge_scores = [
    {"name": "Dave",  "correct": 1, "total": 3},   # should be 33.33%
    {"name": "Eve",   "correct": 2, "total": 3},   # should be 66.67%
    {"name": "Frank", "correct": 0, "total": 5},   # should be 0%
]

results2 = compute_percentages(edge_scores)
for r in results2:
    print(f"  {r['name']}: {r['pct']}%")

print(f"\nDave should be 33.33%, got {results2[0]['pct']}%")
print(f"Eve  should be 66.67%, got {results2[1]['pct']}%")

  Dave: 33%
  Eve: 66%
  Frank: 0%

Dave should be 33.33%, got 33%
Eve  should be 66.67%, got 66%


### Step 1 — REPRODUCE

No traceback. The code runs. But `int()` **truncates** — Dave gets 33 instead of 33.33. In a report that rounds to display, this might pass unnoticed. In a grading system that uses thresholds (e.g., 33.33% ≥ 33.3 to pass), truncation causes wrong pass/fail decisions.

### Step 2 — LOCALISE

In [10]:
# Localise: trace what int() does
raw_value = 1 / 3 * 100
truncated = int(raw_value)
rounded   = round(raw_value, 2)

print(f"Raw:       {raw_value}")
print(f"int():     {truncated}   ← always truncates toward zero")
print(f"round():   {rounded}     ← preserves precision")

Raw:       33.33333333333333
int():     33   ← always truncates toward zero
round():   33.33     ← preserves precision


**Root cause:** `int()` truncates toward zero. This is a silent precision loss.

### Step 3 — SHRINK

Minimal case: `int(1/3 * 100)` → `33` instead of `33.33`.

### Step 4 — CHANGE ONE THING

Replace `int(...)` with `round(..., 2)` to keep the fractional part.

In [11]:
def compute_percentages_fixed(data):
    """Add a 'pct' field — using round() instead of int()."""
    for row in data:
        row["pct"] = round(row["correct"] / row["total"] * 100, 2)  # ← ONE change
    return data

edge_scores_2 = [
    {"name": "Dave",  "correct": 1, "total": 3},
    {"name": "Eve",   "correct": 2, "total": 3},
    {"name": "Frank", "correct": 0, "total": 5},
]

results_fixed = compute_percentages_fixed(edge_scores_2)
for r in results_fixed:
    print(f"  {r['name']}: {r['pct']}%")

  Dave: 33.33%
  Eve: 66.67%
  Frank: 0.0%


In [12]:
debug_log.append({
    "error": "Silent: percentages silently truncated (33 instead of 33.33)",
    "cause": "int() truncates toward zero, silently losing the fractional part",
    "fix": "Use round(value, 2) instead of int(value) to preserve precision",
})

---

## Failure 3 — Wrong Merge Key (Confident, Wrong Output)

This is the scariest kind: the code joins two datasets, produces a plausible-looking table, and every row is **wrong**. No error, no empty result — just wrong data presented confidently.

### The Buggy Code

In [13]:
# Two datasets to join
users = [
    {"user_id": 1, "name": "Alice"},
    {"user_id": 2, "name": "Bob"},
    {"user_id": 3, "name": "Carol"},
]

purchases = [
    {"purchase_id": 1, "user_id": 3, "item": "Laptop",  "amount": 999},
    {"purchase_id": 2, "user_id": 1, "item": "Mouse",   "amount": 25},
    {"purchase_id": 3, "user_id": 2, "item": "Keyboard", "amount": 75},
]

def merge_by_position(users, purchases):
    """Merge users and purchases — BUG: matches by list position, not by user_id."""
    merged = []
    for i, user in enumerate(users):
        purchase = purchases[i]
        merged.append({
            "name": user["name"],
            "item": purchase["item"],
            "amount": purchase["amount"],
        })
    return merged

report = merge_by_position(users, purchases)
print("Purchase Report:")
for row in report:
    print(f"  {row['name']} bought {row['item']} for ${row['amount']}")

Purchase Report:
  Alice bought Laptop for $999
  Bob bought Mouse for $25
  Carol bought Keyboard for $75


That looks clean. Three names, three purchases, no errors. But is it **correct**?

### Step 1 — REPRODUCE

> *"Can I demonstrate the output is wrong?"*

We need to check against the source data. Carol (`user_id=3`) bought the Laptop. Alice (`user_id=1`) bought the Mouse. Let's verify.

In [14]:
# Check: who ACTUALLY bought the Laptop?
laptop_purchase = [p for p in purchases if p["item"] == "Laptop"][0]
laptop_buyer_id = laptop_purchase["user_id"]
laptop_buyer = [u for u in users if u["user_id"] == laptop_buyer_id][0]

print(f"Laptop was purchased by user_id={laptop_buyer_id} → {laptop_buyer['name']}")
print(f"Report says: {report[0]['name']} bought {report[0]['item']}")
print()

if report[0]["name"] != laptop_buyer["name"]:
    print("✅ Reproduced! The report matched by LIST POSITION, not by user_id.")
    print("   Every row is potentially wrong — and there was no error at all.")

Laptop was purchased by user_id=3 → Carol
Report says: Alice bought Laptop

✅ Reproduced! The report matched by LIST POSITION, not by user_id.
   Every row is potentially wrong — and there was no error at all.


### Step 2 — LOCALISE

The merge function uses `purchases[i]` — it assumes the i-th user matches the i-th purchase. That's only true if the lists are in the same order by `user_id`, which they aren't.

### Step 3 — SHRINK

In [15]:
# Minimal case: 2 users, 2 purchases in different order
u = [{"user_id": 1, "name": "Alice"}, {"user_id": 2, "name": "Bob"}]
p = [{"purchase_id": 1, "user_id": 2, "item": "X", "amount": 10},
     {"purchase_id": 2, "user_id": 1, "item": "Y", "amount": 20}]

bad_merge = merge_by_position(u, p)
print("Position-based merge:")
for row in bad_merge:
    print(f"  {row['name']} → {row['item']}")

print("\n  Alice should get Y (user_id=1), but got X (user_id=2's purchase)")

Position-based merge:
  Alice → X
  Bob → Y

  Alice should get Y (user_id=1), but got X (user_id=2's purchase)


### Step 4 — CHANGE ONE THING

Build a lookup dict keyed on `user_id` instead of matching by position.

In [16]:
def merge_by_key(users, purchases):
    """Merge by user_id — not by list position."""
    purchase_by_uid = {p["user_id"]: p for p in purchases}  # ← ONE change: lookup dict
    merged = []
    for user in users:
        purchase = purchase_by_uid.get(user["user_id"])
        if purchase:
            merged.append({
                "name": user["name"],
                "item": purchase["item"],
                "amount": purchase["amount"],
            })
    return merged

report_fixed = merge_by_key(users, purchases)
print("Fixed Purchase Report:")
for row in report_fixed:
    print(f"  {row['name']} bought {row['item']} for ${row['amount']}")

Fixed Purchase Report:
  Alice bought Mouse for $25
  Bob bought Keyboard for $75
  Carol bought Laptop for $999


In [17]:
debug_log.append({
    "error": "Silent: report looks plausible but every row has wrong user-purchase pairing",
    "cause": "Merge used list position (index i) instead of matching on user_id",
    "fix": "Build a dict keyed on user_id and look up each user's purchase by key",
})

---

## Review the Debug Log

In [18]:
rows = "| # | Error | Cause | Fix |\n|---|-------|-------|-----|\n"
for i, entry in enumerate(debug_log, 1):
    rows += f"| {i} | {entry['error']} | {entry['cause']} | {entry['fix']} |\n"

md(rows)

| # | Error | Cause | Fix |
|---|-------|-------|-----|
| 1 | Silent: filter returns 0 results, no traceback | Quantity field is a string (from CSV); string comparison gives wrong results | Convert to int() before comparing: int(o['quantity']) > 8 |
| 2 | Silent: percentages silently truncated (33 instead of 33.33) | int() truncates toward zero, silently losing the fractional part | Use round(value, 2) instead of int(value) to preserve precision |
| 3 | Silent: report looks plausible but every row has wrong user-purchase pairing | Merge used list position (index i) instead of matching on user_id | Build a dict keyed on user_id and look up each user's purchase by key |


## Quick-Reference: Silent Failure Detection Checklist

When your code runs without errors, ask these questions **before** trusting the output:

| Check | How | What It Catches |
|-------|-----|----------------|
| Is the result empty? | `len(result)` or print it | Empty filters from wrong types |
| Are the types what I expect? | `type(value)` / `repr(value)` | String vs int, float truncation |
| Does a known row match? | Spot-check one row against source | Wrong merge keys, wrong joins |
| Does the total make sense? | Sanity-check sum, count, average | Aggregation bugs, double-counting |

## 🎯 Summary

### Key Takeaways

1. **Silent failures produce no traceback** — the code runs, Python is happy, and the result is wrong; these are harder to catch than crashes
2. **Type coercion is a top cause** — CSV data arrives as strings; comparing `"10" > "8"` (lexicographic) or truncating with `int()` silently corrupts results
3. **Position-based matching is fragile** — joining two lists by index instead of by a shared key produces plausible but wrong output
4. **"Reproduce" for silent bugs means proving the output is wrong** — you need to know the expected answer and compare it to the actual answer
5. **A debug log turns pain into a reference** — recording error → cause → fix for silent failures builds intuition for spotting them faster next time